In [ ]:

from google.colab import files

uploaded = files.upload()  # Select ML_Budget_Dataset_WorldBank_Filled(1).xlsx from your machine

Saving ML_Budget_Dataset_WorldBank_Completed.xlsx to ML_Budget_Dataset_WorldBank_Completed.xlsx


In [ ]:

from google.colab import files

uploaded = files.upload()  # Select ML_Budget_Dataset_WorldBank_Filled(1).xlsx from your machine

Saving database_2026(1).xlsx to database_2026(1).xlsx


In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# 1. Load Dataset
file_path = "ML_Budget_Dataset_WorldBank_Completed.xlsx"
df = pd.read_excel(file_path, sheet_name="Completed_Dataset")

# 2. Define Features (X)
feature_cols = [
    'population', 'urban_population_percent', 'gdp_usd', 'gdp_per_capita_usd',
    'gdp_growth_percent', 'inflation_percent', 'tax_revenue_usd', 'government_debt_usd',
    'poverty_rate_percent', 'unemployment_rate_percent', 'literacy_rate_percent',
    'life_expectancy_years', 'health_expenditure_usd', 'internet_users_percent',
    'electricity_access_percent', 'co2_emissions', 'corruption_index', 'government_effectiveness'
]

# 3. Target Allocations
sector_targets = [
    'defence_usd', 'infrastructure_usd', 'research_and_development_usd',
    'education_usd', 'healthcare_usd', 'agriculture_usd',
    'rural_development_usd', 'social_welfare_freebies_usd', 'energy_usd',
    'transport_usd', 'housing_urban_development_usd', 'water_sanitation_usd',
    'environment_climate_usd', 'digital_it_usd', 'industry_msme_usd'
]

sdg_percent_targets = [
    'sdg1_no_poverty_percent', 'sdg2_zero_hunger_percent', 'sdg3_good_health_percent',
    'sdg4_quality_education_percent', 'sdg6_clean_water_percent', 'sdg7_clean_energy_percent',
    'sdg8_decent_work_percent', 'sdg9_industry_innovation_percent',
    'sdg11_sustainable_cities_percent', 'sdg13_climate_action_percent', 'sdg16_peace_justice_percent'
]

X = df[feature_cols]

# 4. Hybrid Ensemble Trainer (CatBoost + GradientBoosting)
def evaluate_hybrid_model(X_data, target_cols, dataset_title):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, df[target_cols], test_size=0.20, random_state=42
    )

    results = []

    for target in target_cols:
        # Member Model 1: Gradient Boosting Regressor
        gbr = GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            random_state=42
        )

        # Member Model 2: CatBoost Regressor
        cb = CatBoostRegressor(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            random_seed=42,
            verbose=False
        )

        # Hybrid Voting Regressor (Equal Weighted Blend)
        hybrid_model = VotingRegressor(
            estimators=[('gbr', gbr), ('cb', cb)],
            weights=[0.5, 0.5]
        )

        hybrid_model.fit(X_train, y_train[target])
        y_pred = hybrid_model.predict(X_test)

        # Calculate Metrics
        mae = mean_absolute_error(y_test[target], y_pred)
        rmse = np.sqrt(mean_squared_error(y_test[target], y_pred))
        mape = mean_absolute_percentage_error(y_test[target], y_pred) * 100
        accuracy = max(0, 100 - mape)
        r2 = r2_score(y_test[target], y_pred)

        results.append({
            'Target Name': target.replace('_usd', '').replace('_percent', '').replace('_', ' ').title(),
            'MAE': mae,
            'RMSE': rmse,
            'MAPE (%)': mape,
            'Accuracy (%)': accuracy,
            'R² Score': r2
        })

    res_df = pd.DataFrame(results)
    print(f"\n=================== {dataset_title} ===================")
    print(res_df.to_string(index=False))

    print(f"\n--- Hybrid Model Overall Summary ---")
    print(f"Macro Avg Accuracy Rate : {res_df['Accuracy (%)'].mean():.2f}%")
    print(f"Macro Avg MAPE          : {res_df['MAPE (%)'].mean():.2f}%")
    print(f"Macro Avg R² Score      : {res_df['R² Score'].mean():.4f}")
    return res_df

# Run Hybrid Evaluation
hybrid_sector_df = evaluate_hybrid_model(X, sector_targets, "HYBRID MODEL - 15 SECTORS (USD)")
hybrid_sdg_df = evaluate_hybrid_model(X, sdg_percent_targets, "HYBRID MODEL - 11 SDGS (%)")


=================== HYBRID MODEL - 15 SECTORS (USD) ===================
              Target Name          MAE         RMSE  MAPE (%)  Accuracy (%)  R² Score
                  Defence 4.614329e+09 1.448729e+10 21.520200     78.479800  0.984864
           Infrastructure 1.160111e+09 2.196518e+09  5.495860     94.504140  0.999270
 Research And Development 2.451883e+09 4.894826e+09 52.326475     47.673525  0.996906
                Education 8.631167e+09 2.290775e+10 18.225394     81.774606  0.987879
               Healthcare 8.331066e+09 3.107078e+10 10.830085     89.169915  0.995393
              Agriculture 5.851840e+08 1.131590e+09  5.467443     94.532557  0.999225
        Rural Development 5.231491e+08 1.044302e+09  5.526423     94.473577  0.999138
  Social Welfare Freebies 1.732912e+09 3.334011e+09  5.463592     94.536408  0.999252
                   Energy 7.297603e+08 1.463096e+09  5.498652     94.501348  0.999171
                Transport 8.935824e+08 1.751152e+09  5.511511     9